<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - End-to-End GNN: Multiclass Apply Event Prediction
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:20px;font-family:Arial"><b>Introduction</b></p>
<p style="font-size:16px;font-family:Arial"> In this notebook we will predict which specific product a user will apply for using end-to-end GNN classification.
<ol style="font-size:16px;font-family:Arial"> Target Classes (Dynamically Discovered) 
     <li>Class 0: Other (No Application)</li>
<li>Class 1-N: Individual Apply events found in data</li>
     </ol>
 <ol style="font-size:16px;font-family:Arial">Models
     <li>GraphSAGE - Scalable neighborhood aggregation</li>
<li>GIN - Most expressive (WL-test equivalent)
</li>
     </ol>   
<p style="font-size:18px;font-family:Arial">The overall processing pipeline follows the sequence as below:</p> 
<img src="./images/gnn_multi.png" alt="gnn" style="width:100%; border: 4px solid #404040; border-radius: 10px;" />
<br>

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>1. Setup and Imports </b></p>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import SAGEConv, GINConv, global_mean_pool, global_max_pool, global_add_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle

from teradataml import *

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Using Apple M2 GPU (MPS)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Using CPU")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>2. Robust Label Encoder</b></p>

In [ ]:
class RobustLabelEncoder:
    """Handles unseen labels by mapping to <UNK> (index 0)."""
    
    def __init__(self):
        self.classes_ = None
        self.class_to_idx = None
        
    def fit(self, values):
        unique = sorted(set(values))
        self.classes_ = ['<UNK>'] + list(unique)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes_)}
        return self
    
    def transform(self, values):
        return np.array([self.class_to_idx.get(v, 0) for v in values])
    
    @property
    def vocab_size(self):
        return len(self.classes_)

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud Lake...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=6.1_Bank_ClickStream_-_Outcome_Prediction_Model_with_Full_GNN_Model.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>4. Load Data & Discover Target Classes</b></p>

In [ ]:
tdf = DataFrame(in_schema('DEMO_Bank','Session_Events'))

In [ ]:
TARGET_PREFIX = "Apply"  # Events starting with this prefix become target classes

df = tdf[['UserID', 'SessionID', 'Event']].to_pandas(all_rows = True)

print(f"Shape: {df.shape}")
print(f"Events: {df['Event'].nunique()}")
print(f"Sessions: {df.groupby(['UserID', 'SessionID']).ngroups}")

In [ ]:
# Dynamically discover Apply* events
apply_events = sorted(df[df['Event'].str.startswith(TARGET_PREFIX)]['Event'].unique())

print(f"\nDiscovered {len(apply_events)} target events (prefix: '{TARGET_PREFIX}'):")
for e in apply_events:
    count = (df['Event'] == e).sum()
    print(f"  - {e}: {count}")

In [ ]:
# Build target class mapping dynamically
# Class 0 = "Other" (no application)
# Class 1-N = Individual Apply events

TARGET_CLASSES = ['Other'] + apply_events
TARGET_TO_IDX = {t: i for i, t in enumerate(TARGET_CLASSES)}
IDX_TO_TARGET = {i: t for t, i in TARGET_TO_IDX.items()}
NUM_CLASSES = len(TARGET_CLASSES)

print(f"\nTarget Classes ({NUM_CLASSES}):")
for idx, name in IDX_TO_TARGET.items():
    print(f"  {idx}: {name}")

In [ ]:
def get_session_target(session_events, target_to_idx):
    """
    Get target class for a session.
    Returns the FIRST Apply event found, or 0 (Other) if none.
    """
    for event in session_events:
        if event in target_to_idx:
            return target_to_idx[event]
    return 0  # Other


# Create session-level targets
session_targets = []
for (uid, sid), group in df.groupby(['UserID', 'SessionID'], sort=False):
    events = group['Event'].tolist()
    target = get_session_target(events, TARGET_TO_IDX)
    session_targets.append({'UserID': uid, 'SessionID': sid, 'target': target})

session_labels = pd.DataFrame(session_targets)

print("\nTarget Distribution:")
for idx in range(NUM_CLASSES):
    count = (session_labels['target'] == idx).sum()
    pct = count / len(session_labels) * 100
    print(f"  {idx} ({IDX_TO_TARGET[idx]}): {count} ({pct:.2f}%)")

In [ ]:
# Event encoder - exclude Apply events from input features
EXCLUDE_TARGET_FROM_INPUT = True

if EXCLUDE_TARGET_FROM_INPUT:
    input_events = df[~df['Event'].str.startswith(TARGET_PREFIX)]['Event'].unique()
    print(f"Input events (excluding {TARGET_PREFIX}*): {len(input_events)}")
else:
    input_events = df['Event'].unique()
    print(f"Input events (all): {len(input_events)}")

event_encoder = RobustLabelEncoder()
event_encoder.fit(input_events)

EVENT_VOCAB = event_encoder.vocab_size
print(f"Event vocabulary: {EVENT_VOCAB}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>5. Graph Construction</b></p>

In [ ]:
class SimpleGraphBuilder:
    """
    Builds graphs using only events.
    Optionally excludes target events from input.
    """
    
    def __init__(self, event_encoder, target_prefix='Apply', exclude_targets=True):
        self.event_encoder = event_encoder
        self.target_prefix = target_prefix
        self.exclude_targets = exclude_targets
        
    def build_graph(self, session_df, target=0):
        session_df = session_df.reset_index(drop=True)
        
        # Filter out target events if needed
        if self.exclude_targets:
            session_df = session_df[~session_df['Event'].str.startswith(self.target_prefix)].reset_index(drop=True)
        
        n = len(session_df)
        
        if n == 0:
            # Session only had target events - create minimal graph
            return Data(
                x=torch.tensor([[0, 0.0]], dtype=torch.float),
                edge_index=torch.tensor([[0], [0]], dtype=torch.long),
                y=torch.tensor([target], dtype=torch.long),
                num_nodes=1,
                event_ids=torch.tensor([0], dtype=torch.long)
            )
        
        event_ids = self.event_encoder.transform(session_df['Event'].values)
        positions = np.arange(n) / max(n - 1, 1)
        node_features = np.column_stack([event_ids, positions])
        
        if n > 1:
            src = list(range(n-1)) + list(range(1, n))
            dst = list(range(1, n)) + list(range(n-1))
            edge_index = torch.tensor([src, dst], dtype=torch.long)
        else:
            edge_index = torch.tensor([[0], [0]], dtype=torch.long)
        
        data = Data(
            x=torch.tensor(node_features, dtype=torch.float),
            edge_index=edge_index,
            y=torch.tensor([target], dtype=torch.long),
            num_nodes=n
        )
        data.event_ids = torch.tensor(event_ids, dtype=torch.long)
        
        return data


graph_builder = SimpleGraphBuilder(
    event_encoder, 
    target_prefix=TARGET_PREFIX, 
    exclude_targets=EXCLUDE_TARGET_FROM_INPUT
)
print("✅ Graph builder ready")

In [ ]:
# Build all graphs
def build_all_graphs(df, session_labels, graph_builder):
    graphs, labels, session_ids = [], [], []
    grouped = df.groupby(['UserID', 'SessionID'], sort=False)
    target_lookup = session_labels.set_index(['UserID', 'SessionID'])['target'].to_dict()
    
    for (uid, sid), sdf in tqdm(grouped, desc="Building graphs"):
        target = target_lookup.get((uid, sid), 0)
        graph = graph_builder.build_graph(sdf, target)
        if graph is not None:
            graphs.append(graph)
            labels.append(target)
            session_ids.append((uid, sid))
    
    return graphs, labels, session_ids


graphs, labels, session_ids = build_all_graphs(df, session_labels, graph_builder)

print(f"\n✅ Built {len(graphs)} graphs")
print(f"\nClass distribution:")
for idx in range(NUM_CLASSES):
    count = labels.count(idx)
    print(f"  {idx} ({IDX_TO_TARGET[idx]}): {count}")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>6. Model Architectures</b></p>

In [ ]:
class GraphSAGEClassifier(nn.Module):
    """GraphSAGE for multiclass classification."""
    
    def __init__(self, event_vocab, num_classes, embed_dim=64, hidden_dim=64, num_layers=3, dropout=0.3):
        super().__init__()
        
        self.event_emb = nn.Embedding(event_vocab, embed_dim, padding_idx=0)
        input_dim = embed_dim + 1  # embedding + position
        
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(SAGEConv(input_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, data):
        e = self.event_emb(data.event_ids)
        pos = data.x[:, 1:2]
        x = torch.cat([e, pos], dim=1)
        
        edge_index, batch = data.edge_index, data.batch
        
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.classifier(x)

In [ ]:
class GINClassifier(nn.Module):
    """GIN for multiclass classification - most expressive."""
    
    def __init__(self, event_vocab, num_classes, embed_dim=64, hidden_dim=64, num_layers=3, dropout=0.3):
        super().__init__()
        
        self.event_emb = nn.Embedding(event_vocab, embed_dim, padding_idx=0)
        input_dim = embed_dim + 1
        self.num_layers = num_layers
        
        def make_mlp(in_d, out_d):
            return nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Linear(out_d, out_d)
            )
        
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(GINConv(make_mlp(input_dim, hidden_dim)))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        for _ in range(num_layers - 1):
            self.convs.append(GINConv(make_mlp(hidden_dim, hidden_dim)))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * num_layers, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, data):
        e = self.event_emb(data.event_ids)
        pos = data.x[:, 1:2]
        x = torch.cat([e, pos], dim=1)
        
        edge_index, batch = data.edge_index, data.batch
        
        layer_outs = []
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
            layer_outs.append(global_add_pool(x, batch))
        
        x = torch.cat(layer_outs, dim=1)  # Jumping Knowledge
        return self.classifier(x)

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>7. Training</b></p>

In [ ]:
# Data splits (stratified)
train_idx, test_idx = train_test_split(
    range(len(graphs)), test_size=0.2, stratify=labels, random_state=SEED
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, stratify=[labels[i] for i in train_idx], random_state=SEED
)

train_graphs = [graphs[i] for i in train_idx]
val_graphs = [graphs[i] for i in val_idx]
test_graphs = [graphs[i] for i in test_idx]

BATCH_SIZE = 64
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)

print(f"Train: {len(train_graphs)} | Val: {len(val_graphs)} | Test: {len(test_graphs)}")

In [ ]:
# Class weights for imbalanced data
train_labels = [g.y.item() for g in train_graphs]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {i} ({IDX_TO_TARGET[i]}): {w:.4f}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        loss = criterion(logits, batch.y)
        probs = F.softmax(logits, dim=1)
        
        total_loss += loss.item() * batch.num_graphs
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
        
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    return {
        'loss': total_loss / total,
        'accuracy': correct / total,
        'preds': np.array(all_preds),
        'labels': np.array(all_labels),
        'probs': np.array(all_probs)
    }

In [ ]:
def train_model(model, train_loader, val_loader, device, num_classes, epochs=5, lr=1e-3, patience=1):
    """Full training loop for multiclass."""
    
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    
    best_acc = 0
    counter = 0
    best_state = None
    history = {'val_acc': []}
    
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val = evaluate(model, val_loader, criterion, device)
        
        history['val_acc'].append(val['accuracy'])
        scheduler.step(val['accuracy'])
        
        if val['accuracy'] > best_acc:
            best_acc = val['accuracy']
            counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {train_loss:.4f} | Val Acc: {val['accuracy']:.4f}")
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_state)
    return model, history, best_acc

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>8. Train & Compare Models</b></p>

In [ ]:
results = {}

# GraphSAGE
print("="*50 + "\nTraining GraphSAGE (Multiclass)\n" + "="*50)
sage = GraphSAGEClassifier(EVENT_VOCAB, NUM_CLASSES).to(DEVICE)
sage, sage_hist, _ = train_model(sage, train_loader, val_loader, DEVICE, NUM_CLASSES)
sage_test = evaluate(sage, test_loader, nn.CrossEntropyLoss(weight=class_weights_tensor), DEVICE)
results['GraphSAGE'] = {'model': sage, 'test': sage_test, 'history': sage_hist}
print(f"✅ Test Accuracy: {sage_test['accuracy']:.4f}")

In [ ]:
# GIN
print("\n" + "="*50 + "\nTraining GIN (Multiclass)\n" + "="*50)
gin = GINClassifier(EVENT_VOCAB, NUM_CLASSES).to(DEVICE)
gin, gin_hist, _ = train_model(gin, train_loader, val_loader, DEVICE, NUM_CLASSES)
gin_test = evaluate(gin, test_loader, nn.CrossEntropyLoss(weight=class_weights_tensor), DEVICE)
results['GIN'] = {'model': gin, 'test': gin_test, 'history': gin_hist}
print(f"✅ Test Accuracy: {gin_test['accuracy']:.4f}")

In [ ]:
# Compare
print("\n" + "="*50 + "\nMODEL COMPARISON\n" + "="*50)
for name, res in results.items():
    print(f"{name:12s} | Accuracy: {res['test']['accuracy']:.4f}")

best_name = max(results, key=lambda k: results[k]['test']['accuracy'])
print(f"\n🏆 Best: {best_name}")

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
for name, res in results.items():
    ax.plot(res['history']['val_acc'], label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Model Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('multiclass_comparison.png', dpi=150)
plt.show()

In [ ]:
# Best model detailed results
best = results[best_name]

print(f"\n{best_name} - Classification Report:")
print(classification_report(
    best['test']['labels'], 
    best['test']['preds'],
    target_names=[IDX_TO_TARGET[i] for i in range(NUM_CLASSES)],
    zero_division=0
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(best['test']['labels'], best['test']['preds'])

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[IDX_TO_TARGET[i] for i in range(NUM_CLASSES)],
    yticklabels=[IDX_TO_TARGET[i] for i in range(NUM_CLASSES)]
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'{best_name} - Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_multiclass.png', dpi=150)
plt.show()

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>9. Save Best Model</b></p>

In [ ]:
torch.save({
    'model_state_dict': best['model'].state_dict(),
    'model_type': best_name,
    'config': {
        'event_vocab': EVENT_VOCAB,
        'num_classes': NUM_CLASSES,
        'embed_dim': 64,
        'hidden_dim': 64,
        'num_layers': 3,
        'dropout': 0.3
    }
}, 'best_e2e_multiclass.pt')

with open('multiclass_e2e_config.pkl', 'wb') as f:
    pickle.dump({
        'event_encoder': event_encoder,
        'target_classes': TARGET_CLASSES,
        'target_to_idx': TARGET_TO_IDX,
        'idx_to_target': IDX_TO_TARGET,
        'target_prefix': TARGET_PREFIX,
        'exclude_targets': EXCLUDE_TARGET_FROM_INPUT
    }, f)

print(f"✅ Saved {best_name}:")
print("   - best_e2e_multiclass.pt")
print("   - multiclass_e2e_config.pkl")

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>10. Inference Pipeline</b></p>

In [ ]:
class MulticlassE2EPipeline:
    """
    End-to-end inference for multiclass Apply prediction.
    """
    
    def __init__(self, model_path, config_path, device='cpu'):
        self.device = torch.device(device)
        
        # Load config
        with open(config_path, 'rb') as f:
            config = pickle.load(f)
        
        self.event_encoder = config['event_encoder']
        self.target_classes = config['target_classes']
        self.idx_to_target = config['idx_to_target']
        self.target_prefix = config['target_prefix']
        self.exclude_targets = config['exclude_targets']
        
        # Load model
        ckpt = torch.load(model_path, map_location=device)
        cfg = ckpt['config']
        model_type = ckpt['model_type']
        
        if model_type == 'GraphSAGE':
            self.model = GraphSAGEClassifier(
                cfg['event_vocab'], cfg['num_classes'],
                cfg['embed_dim'], cfg['hidden_dim'], cfg['num_layers'], cfg['dropout']
            )
        else:  # GIN
            self.model = GINClassifier(
                cfg['event_vocab'], cfg['num_classes'],
                cfg['embed_dim'], cfg['hidden_dim'], cfg['num_layers'], cfg['dropout']
            )
        
        self.model.load_state_dict(ckpt['model_state_dict'])
        self.model.to(self.device).eval()
        
        self.graph_builder = SimpleGraphBuilder(
            self.event_encoder, 
            target_prefix=self.target_prefix,
            exclude_targets=self.exclude_targets
        )
        
        print(f"✅ Loaded {model_type} ({len(self.target_classes)} classes)")
    
    @torch.no_grad()
    def predict(self, events):
        """
        Predict which product user will apply for.
        
        Args:
            events: list of event names
        
        Returns:
            dict with prediction, confidence, and all probabilities
        """
        # Filter out target events
        if self.exclude_targets:
            events = [e for e in events if not e.startswith(self.target_prefix)]
        
        # Check unseen
        known = set(self.event_encoder.classes_)
        unseen = set(events) - known
        if unseen:
            print(f"⚠️ Unseen events → <UNK>: {unseen}")
        
        # Build graph
        session_df = pd.DataFrame({'Event': events})
        graph = self.graph_builder.build_graph(session_df)
        
        # Predict
        batch = next(iter(DataLoader([graph], batch_size=1))).to(self.device)
        logits = self.model(batch)
        probs = F.softmax(logits, dim=1)[0].cpu().numpy()
        
        pred_idx = probs.argmax()
        
        return {
            'prediction': self.idx_to_target[pred_idx],
            'confidence': float(probs[pred_idx]),
            'probabilities': {self.idx_to_target[i]: float(p) for i, p in enumerate(probs)}
        }
    
    def predict_top_k(self, events, k=3):
        """
        Get top-k predictions.
        """
        result = self.predict(events)
        sorted_probs = sorted(result['probabilities'].items(), key=lambda x: x[1], reverse=True)
        return sorted_probs[:k]


# Initialize
pipeline = MulticlassE2EPipeline(
    'best_e2e_multiclass.pt',
    'multiclass_e2e_config.pkl',
    device='mps' if torch.backends.mps.is_available() else 'cpu'
)

In [ ]:
# Test predictions
print("="*60)
print("TESTING PREDICTIONS")
print("="*60)

# Credit card journey
r = pipeline.predict(['ViewCreditCardRates', 'CreditCardBenefits', 'CompareCreditCards'])
print(f"\n📍 Credit card journey:")
print(f"   Prediction: {r['prediction']}")
print(f"   Confidence: {r['confidence']:.4f}")
print(f"   Top 3:")
for name, prob in pipeline.predict_top_k(['ViewCreditCardRates', 'CreditCardBenefits', 'CompareCreditCards'], k=3):
    print(f"      {name}: {prob:.4f}")

In [ ]:
# Savings journey
r = pipeline.predict(['ViewSavingsOptions', 'CompareSavingsRates', 'SavingsCalculator'])
print(f"\n📍 Savings journey:")
print(f"   Prediction: {r['prediction']}")
print(f"   Confidence: {r['confidence']:.4f}")

In [ ]:
# Auto loan journey
r = pipeline.predict(['ViewAutoLoanOptions', 'AutoLoanCalculator', 'CompareAutoRates'])
print(f"\n📍 Auto loan journey:")
print(f"   Prediction: {r['prediction']}")
print(f"   Confidence: {r['confidence']:.4f}")

In [ ]:
# Mortgage journey
r = pipeline.predict(['ViewMortgageOptions', 'MortgageCalculator', 'CompareMortgageRates', 'DiscussHomeFinancing'])
print(f"\n📍 Mortgage journey:")
print(f"   Prediction: {r['prediction']}")
print(f"   Confidence: {r['confidence']:.4f}")

In [ ]:
# Routine banking (no application expected)
r = pipeline.predict(['AccountOverview', 'ViewChecking', 'PayBills', 'ViewStatements'])
print(f"\n📍 Routine banking:")
print(f"   Prediction: {r['prediction']}")
print(f"   Confidence: {r['confidence']:.4f}")

<hr style="height:1px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>Summary</b></p>

<ol style = 'font-size:16px;font-family:Arial'>In this notebook we had done Dynamic Class Discovery. 
Target classes are automatically discovered from data using the prefix (default: Apply)<br>
    <b>  Classes Found</b><br>
TARGET_PREFIX = "Apply"

<p style = 'font-size:16px;font-family:Arial'><b>Usage</b><code>
result = pipeline.predict(['ViewCreditCardRates', 'CompareCreditCards']) 
print(result['confidence'])  # 0.73 
print(result['probabilities'])  # {'Other': 0.1, 'ApplyCreditCard': 0.73, ...}
#Get top-k predictions
top3 = pipeline.predict_top_k(events, k=3)    
</code>

<p style = 'font-size:18px;font-family:Arial'><b>Key Features</b>
    <ol style = 'font-size:16px;font-family:Arial'>
<li>Dynamic discovery of target classes </li>
<li>Target events excluded from input (no data leakage)  </li>
<li>Class weights for imbalanced data </li>
<li>Handles unseen events via UNK token </li>
        <li>Top-k predictions available </li>

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>